# 🔴 Solution: Multi-Head Attention（参考版）

## 📋 题目描述

**难度：** Hard

从零实现多头注意力。

MHA 将输入投影到多个头，每个头计算缩放点积注意力，然后拼接并投影结果。

**签名:** `MultiHeadAttention(d_model, num_heads)`

**方法:** `forward(Q, K, V) -> Tensor`
- `Q` — 查询张量 (B, S_q, d_model)
- `K` — 键张量 (B, S_k, d_model)
- `V` — 值张量 (B, S_k, d_model)

**返回:** 注意力输出 (B, S_q, d_model)

**约束:**
- 使用 W_q、W_k、W_v、W_o 作为 `nn.Linear(d_model, d_model)`
- `d_k = d_model // num_heads`
- 支持交叉注意力（S_q != S_k）

**提示：** 1. 用 `nn.Linear(d_model, d_model)` 投影 Q/K/V
2. `.view(...).transpose(1,2)` → `(B, heads, S, d_k)`
3. `scores = Q @ K.T / sqrt(d_k)` → `softmax` → `@ V`
4. transpose + reshape → `W_o` 投影


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def multi_head_attention(x: torch.Tensor, W_q: torch.Tensor, W_k: torch.Tensor, W_v: torch.Tensor, W_o: torch.Tensor, num_heads: int, mask: torch.Tensor = None) -> torch.Tensor:
    # Multi-Head Attention：将注意力分成多个头并行计算
    # 直觉：每个头学习不同的注意力模式（语法、语义、位置等）

    batch_size, seq_len, d_model = x.shape
    # ---- Step 1: 计算每个头的维度 ----
    # d_model 必须能被 num_heads 整除
    d_k = d_model // num_heads

    # ---- Step 2: 线性投影得到 Q, K, V ----
    # x: [B, S, d_model] @ W_q: [d_model, d_model] → Q: [B, S, d_model]
    # 每个头的权重被"隐式"包含在同一个大矩阵中
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v

    # ---- Step 3: 拆分成多头 ----
    # [B, S, d_model] → [B, S, num_heads, d_k] → [B, num_heads, S, d_k]
    # transpose(1,2) 把 num_heads 维提前，方便批量做注意力
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)

    # ---- Step 4: 缩放点积注意力 ----
    # scores: [B, num_heads, S, S] — 每个头独立计算注意力
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    # ---- Step 5: Mask（可选） ----
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # ---- Step 6: Softmax + 加权求和 ----
    attn = torch.softmax(scores, dim=-1)
    # attn: [B, H, S, S] @ V: [B, H, S, d_k] → [B, H, S, d_k]
    out = attn @ V

    # ---- Step 7: 合并多头 ----
    # transpose 回来：[B, H, S, d_k] → [B, S, H, d_k]
    # contiguous()：transpose 后内存不连续，需要重新排列
    # view：合并 H 和 d_k 回 d_model
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

    # ---- Step 8: 输出投影 ----
    # 最终线性变换，混合各头的信息
    return out @ W_o

In [ ]:
# Verify
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Self-attn shape:", out.shape)

Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)

In [ ]:
# Run judge
from torch_judge import check
check("mha")

## 📝 核心思路总结

1. **多头拆分**：将 d_model 拆成 num_heads × d_k，各头独立计算注意力
2. **view + transpose**：reshape 的关键操作，注意 contiguous() 的必要性
3. **输出投影**：W_o 混合各头信息，恢复到 d_model 维度
4. **为什么多头**：不同头学习不同的注意力模式